# Demo 07: Held-out test evaluation

Load the adapter saved by Demo 06 and evaluate it only on the held-out test split. The next cell imports the required libraries and defines validation and batched-generation helpers.

In [ ]:
import json
import platform
from pathlib import Path

from IPython.display import Markdown, display
import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


def find_project_root(start_directory: Path) -> Path:
    """Return the repository root containing project instructions."""
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise RuntimeError('Could not find the project root. Start JupyterLab from the repository root.')


def require_bfloat16_mps() -> torch.device:
    """Return MPS or raise an actionable BF16 support error."""
    if not torch.backends.mps.is_built() or not torch.backends.mps.is_available():
        raise RuntimeError('This evaluation requires an available MPS backend; CPU fallback is disabled.')
    macos_version = platform.mac_ver()[0]
    if not macos_version or int(macos_version.split('.')[0]) < 14:
        raise RuntimeError('BF16 MPS evaluation requires macOS 14 or later.')
    return torch.device('mps')


def validate_test_dataset(dataset) -> None:
    """Validate held-out records without printing ticket text."""
    if len(dataset) == 0:
        raise ValueError('The held-out test dataset is empty.')
    for index, record in enumerate(dataset):
        messages = record.get('messages')
        if not isinstance(messages, list) or len(messages) != 2 or [message.get('role') for message in messages] != ['user', 'assistant']:
            raise ValueError(f'Test record {index} must contain user and assistant messages.')
        if not messages[0].get('content', '').strip():
            raise ValueError(f'Test record {index} has empty ticket text.')
        try:
            output = json.loads(messages[1].get('content', ''))
        except json.JSONDecodeError as error:
            raise ValueError(f'Test record {index} has invalid assistant JSON.') from error
        if not isinstance(output, dict) or set(output) != {'category', 'severity', 'summary'}:
            raise ValueError(f'Test record {index} has an invalid output schema.')


def calculate_test_metrics(model, tokenizer, test_dataset) -> dict[str, float]:
    """Generate held-out responses and calculate aggregate classification metrics."""
    valid_json_count = category_correct_count = severity_correct_count = joint_correct_count = 0
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    model.eval()
    try:
        for start_index in range(0, len(test_dataset), TEST_GENERATION_BATCH_SIZE):
            batch_records = test_dataset.select(range(start_index, min(start_index + TEST_GENERATION_BATCH_SIZE, len(test_dataset))))
            prompts = [tokenizer.apply_chat_template([record['messages'][0]], tokenize=False, add_generation_prompt=True, enable_thinking=False) for record in batch_records]
            model_inputs = {name: tensor.to(DEVICE) for name, tensor in tokenizer(prompts, return_tensors='pt', padding=True).items()}
            with torch.inference_mode():
                output_ids = model.generate(**model_inputs, max_new_tokens=GENERATION_MAX_NEW_TOKENS, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
            predictions = tokenizer.batch_decode(output_ids[:, model_inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
            for record, prediction in zip(batch_records, predictions):
                try:
                    generated = json.loads(prediction.strip())
                    valid_json = isinstance(generated, dict)
                except json.JSONDecodeError:
                    generated, valid_json = {}, False
                reference = json.loads(record['messages'][1]['content'])
                category_correct = valid_json and generated.get('category') == reference['category']
                severity_correct = valid_json and generated.get('severity') == reference['severity']
                valid_json_count += valid_json
                category_correct_count += category_correct
                severity_correct_count += severity_correct
                joint_correct_count += category_correct and severity_correct
    finally:
        tokenizer.padding_side = original_padding_side
    total = len(test_dataset)
    return {'category_accuracy': category_correct_count / total, 'severity_accuracy': severity_correct_count / total, 'joint_accuracy': joint_correct_count / total, 'valid_json_rate': valid_json_count / total}


## Local artifacts and configuration

The next cell validates the local base model, Demo 06 adapter, and test file, then sets MPS BF16 and batched-generation options.

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MODEL_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'models' / 'Qwen3-1.7B'
ADAPTER_DIRECTORY = PROJECT_ROOT / 'ticket-classification' / 'artifacts' / 'models' / 'demo06-qwen3-1.7b-ticket-classification-lora' / 'adapter'
TEST_FILE = PROJECT_ROOT / 'ticket-classification' / 'artifacts' / 'datasets' / 'test_dataset.jsonl'
MODEL_DTYPE = torch.bfloat16
GENERATION_MAX_NEW_TOKENS = 128
TEST_GENERATION_BATCH_SIZE = 4

for required_path in (MODEL_DIRECTORY / 'config.json', MODEL_DIRECTORY / 'tokenizer.json', TEST_FILE, ADAPTER_DIRECTORY / 'adapter_config.json'):
    if not required_path.is_file():
        raise FileNotFoundError(f'Missing required local artifact: {required_path}')
if not any((ADAPTER_DIRECTORY / name).is_file() for name in ('adapter_model.safetensors', 'adapter_model.bin')):
    raise FileNotFoundError(f'Missing adapter weights in {ADAPTER_DIRECTORY}. Complete Demo 06 first.')
DEVICE = require_bfloat16_mps()
print(f'Selected device: {DEVICE}')
print(f'Model dtype: {MODEL_DTYPE}')
print(f'Test generation batch size: {TEST_GENERATION_BATCH_SIZE}')


## Load the held-out test split

The next cell loads and validates only `test_dataset.jsonl`. Training and validation data are never loaded.

In [ ]:
test_dataset = load_dataset('json', data_files={'test': str(TEST_FILE)}, split='test')
validate_test_dataset(test_dataset)
print(f'Validated held-out test records: {len(test_dataset)}')


## Load the fine-tuned adapter

The next cell loads Qwen3-1.7B in BF16, attaches the saved LoRA adapter, and confirms MPS evaluation readiness.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIRECTORY, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_DIRECTORY, dtype=MODEL_DTYPE, local_files_only=True).to(DEVICE)
if base_model.config.model_type != 'qwen3':
    raise RuntimeError(f'Expected a Qwen3 base model, got {base_model.config.model_type!r}.')
model = PeftModel.from_pretrained(base_model, ADAPTER_DIRECTORY, local_files_only=True, autocast_adapter_dtype=False).to(DEVICE)
if not isinstance(model, PeftModel) or next(model.parameters()).device.type != 'mps' or next(model.parameters()).dtype != MODEL_DTYPE:
    raise RuntimeError('The PEFT model was not loaded on MPS in BF16.')
model.eval()
print('Fine-tuned model loaded successfully.')


## Evaluate held-out classification quality

The next cell generates every test response and reports Category Accuracy, Severity Accuracy, Category+Severity Accuracy, and Valid JSON Rate. Invalid JSON counts as incorrect for label metrics.

In [ ]:
test_metrics = calculate_test_metrics(model, tokenizer, test_dataset)
table_lines = [
    '| Metric | Held-out test score |',
    '| --- | ---: |',
    f"| Category Accuracy | {test_metrics['category_accuracy']:.1%} |",
    f"| Severity Accuracy | {test_metrics['severity_accuracy']:.1%} |",
    f"| Category+Severity Accuracy | {test_metrics['joint_accuracy']:.1%} |",
    f"| Valid JSON Rate | {test_metrics['valid_json_rate']:.1%} |",
]
display(Markdown('\n'.join(table_lines)))
